In [1]:
from pyspark.sql import functions as F

GOLD_PATH = "abfss://gold@flightdatalakegen2.dfs.core.windows.net/flight_facts/"

df = spark.read.parquet(GOLD_PATH)

StatementMeta(sparkpool1, 25, 2, Finished, Available, Finished, False)

In [3]:
dim_date = df.select("FL_DATE").distinct()
display(dim_date)

StatementMeta(sparkpool1, 25, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c945242a-86a9-4e85-949c-9f1ddba92cbf)

In [5]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

w = Window.orderBy("FL_DATE")

dim_date = dim_date.withColumn("DateKey", F.row_number().over(w))

dim_date = dim_date \
    .withColumn("Year", F.year("FL_DATE")) \
    .withColumn("Month", F.month("FL_DATE")) \
    .withColumn("Day", F.dayofmonth("FL_DATE")) \
    .withColumn("DayOfWeek", F.dayofweek("FL_DATE")) \
    .withColumn("IsWeekend", F.when(F.dayofweek("FL_DATE").isin(1,7), 1).otherwise(0))

dim_date.write.mode("overwrite") \
    .parquet("abfss://gold@flightdatalakegen2.dfs.core.windows.net/dim_date/")

StatementMeta(sparkpool1, 25, 6, Finished, Available, Finished, False)

In [6]:
dim_airline = df.select("OP_CARRIER").distinct()

w = Window.orderBy("OP_CARRIER")

dim_airline = dim_airline.withColumn(
    "CarrierKey",
    F.row_number().over(w)
)

dim_airline.write.mode("overwrite") \
    .parquet("abfss://gold@flightdatalakegen2.dfs.core.windows.net/dim_airline/")

StatementMeta(sparkpool1, 25, 7, Finished, Available, Finished, False)

In [7]:
airports = df.select(F.col("ORIGIN").alias("AIRPORT")).union(
    df.select(F.col("DEST").alias("AIRPORT"))
).distinct()

w = Window.orderBy("AIRPORT")

dim_airport = airports.withColumn(
    "AirportKey",
    F.row_number().over(w)
)

dim_airport.write.mode("overwrite") \
    .parquet("abfss://gold@flightdatalakegen2.dfs.core.windows.net/dim_airport/")

StatementMeta(sparkpool1, 25, 8, Finished, Available, Finished, False)

In [8]:
dim_route = df.select("ROUTE", "DISTANCE", "HAUL_TYPE").distinct()

w = Window.orderBy("ROUTE")

dim_route = dim_route.withColumn(
    "RouteKey",
    F.row_number().over(w)
)

dim_route.write.mode("overwrite") \
    .parquet("abfss://gold@flightdatalakegen2.dfs.core.windows.net/dim_route/")

StatementMeta(sparkpool1, 25, 9, Finished, Available, Finished, False)

In [9]:
dim_cause = df.select("DELAY_CAUSE").distinct()

w = Window.orderBy("DELAY_CAUSE")

dim_cause = dim_cause.withColumn(
    "CauseKey",
    F.row_number().over(w)
)

dim_cause.write.mode("overwrite") \
    .parquet("abfss://gold@flightdatalakegen2.dfs.core.windows.net/dim_delaycause/")

StatementMeta(sparkpool1, 25, 10, Finished, Available, Finished, False)

In [ ]:
fact = df.select(
    "FL_DATE",
    "OP_CARRIER",
    "ORIGIN",
    "DEST",
    "ROUTE",
    "DELAY_CAUSE",   
    "DEP_DELAY",
    "ARR_DELAY",
    "DISTANCE",
    "AIR_TIME",
    "IS_DELAYED",
    "OTP_FLAG",
    "TOTAL_CAUSE_DELAY"
)

StatementMeta(sparkpool1, 25, 14, Finished, Available, Finished, False)

In [14]:
dim_date = dim_date.select("FL_DATE", "DateKey")
dim_airline = dim_airline.select("OP_CARRIER", "CarrierKey")
dim_airport = dim_airport.select("AIRPORT", "AirportKey")
dim_route = dim_route.select("ROUTE", "RouteKey")
dim_cause = dim_cause.select("DELAY_CAUSE", "CauseKey")

StatementMeta(sparkpool1, 25, 15, Finished, Available, Finished, False)

In [15]:
fact = fact \
    .join(dim_date, "FL_DATE", "left") \
    .join(dim_airline, "OP_CARRIER", "left") \
    .join(dim_route, "ROUTE", "left") \
    .join(dim_cause, "DELAY_CAUSE", "left")

StatementMeta(sparkpool1, 25, 16, Finished, Available, Finished, False)

In [16]:
fact_final = fact.select(
    "DateKey",
    "CarrierKey",
    "RouteKey",
    "CauseKey",
    "ORIGIN",
    "DEST",
    "DEP_DELAY",
    "ARR_DELAY",
    "DISTANCE",
    "AIR_TIME",
    "IS_DELAYED",
    "OTP_FLAG",
    "TOTAL_CAUSE_DELAY"
)

StatementMeta(sparkpool1, 25, 17, Finished, Available, Finished, False)

In [17]:
fact_final.write.mode("overwrite") \
    .parquet("abfss://gold@flightdatalakegen2.dfs.core.windows.net/fact_flight/")

StatementMeta(sparkpool1, 25, 18, Finished, Available, Finished, False)